In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_IMPORTS

In [0]:
%run /Users/arnaldo.jbj@gmail.com/ElectricalGeneratedDistribution-Project/pipelines/02_extracting_from_bronze_to_treatments_into_silver/step00_AUXILIARY_FUNCTIONS

In [0]:
spark.read.table('bronze.contratos_uc').show(5)

In [0]:
spark.read.table('bronze.contratos_uc').printSchema()

In [0]:
TABLE_ORIGIN = 'contratos_uc'

TABLE_DESTINY = 'contracts_uc'

df = (
        spark.read.table(f"bronze.{TABLE_ORIGIN}")
             .withColumnsRenamed(
                                  {
                                    'id_contrato' : 'contract_id',
                                    'id_cliente' : 'customer_id',
                                    'distribuidora' : 'distributor',
                                    'id_usina' : 'plant_id',
                                    'subgrupo' : 'subgroup',
                                    'cota_kwh_mes' : 'month_limit_kwh',
                                    'desconto' : 'discount_pct',
                                    'data_inicio' : 'start_date',
                                    'data_fim' : 'end_date',
                                  }
                                )
             .withColumn('distributor', F.upper(F.col('distributor')))
             .withColumn('uf', F.upper(F.col('uf')))
             .withColumn('regional', F.upper(F.col('regional')))
             .withColumn(
                          'start_date', 
                          F.date_format(
                                         F.to_date(
                                                    F.regexp_replace(F.col('start_date'), r'(\d{2})/(\d{2})/(\d{4})', '$3-$2-$1'),
                                                    'yyyy-MM-dd'
                                                  ), 
                                         'yyyy-MM-dd')
                        )
             .withColumn(
                          'end_date', 
                          F.date_format(
                                         F.to_date(
                                                    F.regexp_replace(F.col('end_date'), r'(\d{2})/(\d{2})/(\d{4})', '$3-$2-$1'),
                                                    'yyyy-MM-dd'
                                                  ), 
                                         'yyyy-MM-dd')
                        )
             .withColumn('discount_pct', F.regexp_replace(F.regexp_replace(F.col('discount_pct'), '%', ''), ',', '.').cast('decimal(5,2)'))
             .withColumn('discount_pct', F.when(F.col('discount_pct') >= 1, F.col('discount_pct') / 100).otherwise(F.col('discount_pct')))
             .withColumn('status', F.upper(F.regexp_replace(F.trim(F.col('status')), '\s+', ' ')))
             .withColumn(
                          'status', 
                          F.when(F.col('status').rlike(r'(?i)^ATIV[AO]S?$|^ACTIVE$'), F.lit('ATIVO'))
                           .when(F.col('status').rlike(r'(?i)^CANCELAD[AO]S?$|^CANCELLED$|^CANCEL$'), F.lit('CANCELADO'))
                           .otherwise(F.col('status')))
             .withColumn('month_limit_kwh', F.col('month_limit_kwh').cast('float'))
             
     )

In [0]:
display(null_or_blank(df))

'end_date' null means the contract is still active. No problem.

In [0]:
display(check_duplicates(df,'contract_id'))

In [0]:
df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"silver.{TABLE_DESTINY}")